File: health-metrics-trends.ipynb

Description: Connects to the real HealthTrack Pro SQLite database and computes real summary statistics and early-vs-late trends for logged personal health metrics (blood pressure, weight, sleep, mood, glucose, etc.), plus a comparison against the on-disk backup snapshots.

Author: Jose-Jorge HERNANDEZ

Company: Parlee Conseiller, Inc.

Date: 2026-07-30

Last edit date: 2026-07-30

Version: 1.0.0

---

# HealthTrack Pro — Health Metrics Trends

## Context & Setup

HealthTrack Pro is a personal desktop app that logs real personal health readings (blood pressure, heart rate, weight, sleep, mood, glucose, etc.) into a local SQLite database, and separately keeps periodic backup snapshots of that database file. Before computing anything, we need to know what's actually in the live database: which tables exist, how many rows each holds, and where the data physically lives on disk. This first cell opens a read-only connection to the real database file and lists its tables and row counts, so every later number in this notebook is traceable back to the live data rather than assumed.

In [1]:
import sqlite3
import os
import re
from datetime import datetime

# Absolute paths to the real, live artifacts on this machine -- no mock/sample data is used anywhere in this notebook.
DB_PATH = r"H:\DEV\HealthTrack Pro\database\healthtrack.db"
BACKUPS_DIR = r"H:\DEV\HealthTrack Pro\backups"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# List every user table in the database (sqlite_master is SQLite's built-in schema catalog).
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]

print(f"Database file : {DB_PATH}")
print(f"Tables found  : {len(tables)} -> {tables}\n")

for t in tables:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:<24s} {cur.fetchone()[0]:>5d} rows")

Database file : H:\DEV\HealthTrack Pro\database\healthtrack.db
Tables found  : 3 -> ['alertas', 'configuracion_usuario', 'registros_salud']

  alertas                    161 rows
  configuracion_usuario        0 rows
  registros_salud            119 rows


## Method

The database turns out to hold three tables: `registros_salud` (the actual per-entry health readings), `alertas` (health alerts the app generated from those readings), and `configuracion_usuario` (app settings, empty). Before summarizing any column we print the real schema (`PRAGMA table_info`) rather than guessing column names, since the app is Spanish-language and the fields (e.g. `presion_sistolica`, `glucosa`, `horas_sueno`) are not self-evident from the app description alone. `pandas`/`matplotlib` are not installed in this environment, so the analysis below uses plain `sqlite3` + Python standard library (`statistics`-free manual aggregation via SQL `MIN`/`MAX`/`AVG`/`COUNT`) — this keeps every number a direct, auditable SQL result rather than a library-computed derivative.

For each numeric health metric we compute: how many entries actually have a value (many optional fields are sparsely filled), the observed min/max, and the average. We then split the logging period at its chronological midpoint and compare early-period vs. late-period averages for the core vitals, to see whether there's a real directional trend in the data (not just a snapshot). Finally we cross-check the `alertas` table (what the app itself flagged as noteworthy) and the backup snapshots on disk, since both are real artifacts of how this data has actually been used over time.

## Results

### Schema

We print the real column list for each table (name, declared type, nullability, primary key) so the rest of the notebook is grounded in the actual schema rather than assumptions from the app's description.

In [2]:
def print_schema(table):
    """Print PRAGMA table_info() for a table: real column names/types/constraints from the live DB."""
    cur.execute(f"PRAGMA table_info({table})")
    cols = cur.fetchall()
    print(f"{table} ({len(cols)} columns):")
    for cid, name, ctype, notnull, default, pk in cols:
        flags = []
        if pk:
            flags.append("PK")
        if notnull:
            flags.append("NOT NULL")
        flag_s = f"  [{', '.join(flags)}]" if flags else ""
        print(f"  - {name:<24s} {ctype:<12s}{flag_s}")
    print()

for t in tables:
    print_schema(t)

alertas (12 columns):
  - id                       INTEGER       [PK, NOT NULL]
  - registro_id              INTEGER     
  - fecha                    DATE          [NOT NULL]
  - metrica                  VARCHAR(100)  [NOT NULL]
  - valor                    VARCHAR(50) 
  - criticidad               VARCHAR(20)   [NOT NULL]
  - titulo                   VARCHAR(200)  [NOT NULL]
  - descripcion              TEXT        
  - recomendacion            TEXT        
  - vista                    BOOLEAN       [NOT NULL]
  - resuelta                 BOOLEAN       [NOT NULL]
  - fecha_creacion           DATETIME      [NOT NULL]

configuracion_usuario (5 columns):
  - id                       INTEGER       [PK, NOT NULL]
  - clave                    VARCHAR(100)  [NOT NULL]
  - valor                    TEXT          [NOT NULL]
  - descripcion              TEXT        
  - fecha_actualizacion      DATETIME      [NOT NULL]

registros_salud (27 columns):
  - id                       INTEGER       [P

### Logging coverage

Before trusting any average, we need to know how much real data backs it: the actual date range covered, how many distinct calendar days have at least one logged entry (vs. gaps), and how entries split across the day (`periodo`: manana/tarde/noche). This tells us whether the "trend" computed later is based on dense, regular logging or sparse, irregular logging.

In [3]:
cur.execute("SELECT MIN(fecha), MAX(fecha), COUNT(DISTINCT fecha) FROM registros_salud")
min_d, max_d, distinct_days = cur.fetchone()
cur.execute("SELECT COUNT(*) FROM registros_salud")
total_records = cur.fetchone()[0]

d0 = datetime.strptime(min_d, "%Y-%m-%d")
d1 = datetime.strptime(max_d, "%Y-%m-%d")
span_days = (d1 - d0).days + 1

print(f"Logging span                : {min_d} to {max_d}  ({span_days} calendar days)")
print(f"Distinct days with an entry  : {distinct_days}  ({distinct_days / span_days * 100:.1f}% of the span)")
print(f"Total logged entries         : {total_records}  (avg {total_records / distinct_days:.2f} entries per logged day)\n")

cur.execute("SELECT periodo, COUNT(*) FROM registros_salud GROUP BY periodo ORDER BY COUNT(*) DESC")
print("Entries by time of day (periodo):")
for periodo, cnt in cur.fetchall():
    print(f"  {periodo:<10s} {cnt:>4d}")

Logging span                : 2026-03-23 to 2026-05-21  (60 calendar days)
Distinct days with an entry  : 57  (95.0% of the span)
Total logged entries         : 119  (avg 2.09 entries per logged day)

Entries by time of day (periodo):
  tarde        44
  noche        40
  manana       35


### Per-metric summary statistics

`registros_salud` has many optional numeric columns (vitals, activity, sleep, mood, glucose, hydration, caffeine). Each is populated a different number of times depending on what the user actually chose to log that day, so the count column below matters as much as the average -- an average from 32 readings (`temperatura_corporal`) carries far less weight than one from all 119 (`presion_sistolica`). We compute count/min/max/avg directly in SQL for every metric that has at least one non-null value.

In [4]:
# Human-readable labels for the Spanish column names, purely for display -- the query itself uses the real column names.
NUMERIC_METRICS = {
    "presion_sistolica": "Systolic BP (mmHg)",
    "presion_diastolica": "Diastolic BP (mmHg)",
    "ritmo_cardiaco": "Heart rate (bpm)",
    "oxigenacion": "Blood oxygen (%)",
    "peso": "Weight (kg)",
    "altura": "Height (cm)",
    "imc": "BMI",
    "pasos": "Steps",
    "distancia_caminada": "Distance walked (km)",
    "calorias_quemadas": "Calories burned",
    "horas_sueno": "Sleep hours",
    "calidad_sueno": "Sleep quality (1-10)",
    "nivel_estres": "Stress level (1-10)",
    "estado_animo": "Mood (1-10)",
    "glucosa": "Glucose (mg/dL)",
    "temperatura_corporal": "Body temperature (C)",
    "consumo_agua": "Water intake (L)",
    "cafeina": "Caffeine (mg)",
}

print(f"{'Metric':<24s}{'n':>6s}{'min':>10s}{'max':>10s}{'avg':>10s}")
print("-" * 60)
for col, label in NUMERIC_METRICS.items():
    cur.execute(f"SELECT COUNT({col}), MIN({col}), MAX({col}), AVG({col}) FROM registros_salud WHERE {col} IS NOT NULL")
    n, mn, mx, avg = cur.fetchone()
    avg_s = f"{avg:.1f}" if avg is not None else "-"
    mn_s = f"{mn}" if mn is not None else "-"
    mx_s = f"{mx}" if mx is not None else "-"
    print(f"{label:<24s}{n:>6d}{mn_s:>10s}{mx_s:>10s}{avg_s:>10s}")

Metric                       n       min       max       avg
------------------------------------------------------------
Systolic BP (mmHg)         119        94       161     127.4
Diastolic BP (mmHg)        119        61       103      80.4
Heart rate (bpm)           119        50        94      71.4
Blood oxygen (%)           119      94.9     100.0      97.6
Weight (kg)                119      75.1      79.5      76.9
Height (cm)                119     170.0     170.0     170.0
BMI                        119      26.0      27.5      26.6
Steps                      119      2963     11895    8081.2
Distance walked (km)       119       2.0       8.8       5.2
Calories burned            119      1492      2748    2155.1
Sleep hours                 40       5.4       9.2       7.1
Sleep quality (1-10)        40         5         9       6.8
Stress level (1-10)        119         2         8       4.6
Mood (1-10)                119         5         9       7.0
Glucose (mg/dL)         

### Free-text logging fields

Three columns (`medicamentos`, `sintomas`, `ejercicio_realizado`) exist in the schema for medications, symptoms, and exercise notes, plus `notas_medicas` for general medical notes. We check how many entries actually used each one, since an unused column changes what conclusions can be drawn (e.g. "no symptoms logged" only means something if the field was ever used at all).

In [5]:
for col in ["medicamentos", "notas_medicas", "sintomas", "ejercicio_realizado"]:
    cur.execute(f"SELECT COUNT(*) FROM registros_salud WHERE {col} IS NOT NULL AND TRIM({col}) <> ''")
    n = cur.fetchone()[0]
    print(f"{col:<22s} used in {n:>3d} / {total_records} entries")

medicamentos           used in   0 / 119 entries
notas_medicas          used in  30 / 119 entries
sintomas               used in   0 / 119 entries
ejercicio_realizado    used in   0 / 119 entries


### Early-vs-late trend

A single average hides direction. To see whether the core vitals moved over the ~2 months of logging, we split all entries at their chronological midpoint (by date, not by row id, since rows aren't necessarily inserted in date order) and compare the average of each core metric before vs. on/after that midpoint. This is a real, data-derived trend check, not a fitted model -- appropriate given the modest sample size (119 entries).

In [6]:
cur.execute("SELECT fecha FROM registros_salud ORDER BY fecha")
all_dates = [r[0] for r in cur.fetchall()]
midpoint_date = all_dates[len(all_dates) // 2]

TREND_METRICS = {
    "presion_sistolica": "Systolic BP",
    "presion_diastolica": "Diastolic BP",
    "ritmo_cardiaco": "Heart rate",
    "peso": "Weight",
    "nivel_estres": "Stress level",
    "estado_animo": "Mood",
    "glucosa": "Glucose",
}

print(f"Chronological midpoint: {midpoint_date}  (comparing entries before it vs. on/after it)\n")
print(f"{'Metric':<16s}{'Early avg':>12s}{'Late avg':>12s}{'Change':>10s}")
print("-" * 50)
for col, label in TREND_METRICS.items():
    cur.execute(f"SELECT AVG({col}) FROM registros_salud WHERE fecha < ? AND {col} IS NOT NULL", (midpoint_date,))
    early = cur.fetchone()[0]
    cur.execute(f"SELECT AVG({col}) FROM registros_salud WHERE fecha >= ? AND {col} IS NOT NULL", (midpoint_date,))
    late = cur.fetchone()[0]
    if early is not None and late is not None:
        print(f"{label:<16s}{early:>12.2f}{late:>12.2f}{late - early:>+10.2f}")
    else:
        print(f"{label:<16s}{'n/a':>12s}{'n/a':>12s}{'n/a':>10s}")

Chronological midpoint: 2026-04-25  (comparing entries before it vs. on/after it)

Metric             Early avg    Late avg    Change
--------------------------------------------------
Systolic BP           128.44      126.50     -1.94
Diastolic BP           81.26       79.55     -1.71
Heart rate             71.46       71.29     -0.17
Weight                 77.63       76.30     -1.33
Stress level            4.72        4.48     -0.24
Mood                    7.14        6.95     -0.19
Glucose                97.85       99.13     +1.27


### Alerts generated by the app

The app itself generates `alertas` rows whenever a reading crosses a threshold it cares about. This is a second, independent signal on top of the raw stats above: it tells us which metric the app's own logic considers most concerning, and how many of those alerts have actually been reviewed/resolved by the user.

In [7]:
cur.execute("SELECT COUNT(*) FROM alertas")
total_alerts = cur.fetchone()[0]

cur.execute("SELECT criticidad, COUNT(*) FROM alertas GROUP BY criticidad ORDER BY COUNT(*) DESC")
by_crit = cur.fetchall()

cur.execute("SELECT metrica, COUNT(*) FROM alertas GROUP BY metrica ORDER BY COUNT(*) DESC")
by_metric = cur.fetchall()

cur.execute("SELECT vista, COUNT(*) FROM alertas GROUP BY vista")
by_vista = cur.fetchall()

cur.execute("SELECT resuelta, COUNT(*) FROM alertas GROUP BY resuelta")
by_resuelta = cur.fetchall()

print(f"Total alerts generated: {total_alerts}\n")

print("By criticality:")
for c, n in by_crit:
    print(f"  {c:<12s} {n:>4d}")

print("\nBy triggering metric:")
for m, n in by_metric:
    print(f"  {m:<20s} {n:>4d}")

print("\nViewed (vista: 0=no,1=yes):")
for v, n in by_vista:
    print(f"  {v}: {n}")

print("\nResolved (resuelta: 0=no,1=yes):")
for r, n in by_resuelta:
    print(f"  {r}: {n}")

Total alerts generated: 161

By criticality:
  atencion      127
  preocupante    34

By triggering metric:
  presion_arterial       87
  nivel_estres           29
  glucosa                24
  ritmo_cardiaco         17
  oxigenacion             2
  horas_sueno             2

Viewed (vista: 0=no,1=yes):
  1: 161

Resolved (resuelta: 0=no,1=yes):
  0: 161


### Backup snapshots vs. live data

The app separately keeps timestamped `.db` backup snapshots in `backups\`. We list what's actually on disk there (filename, size, file modified time) and, where the filename encodes a timestamp, compare the snapshot dates against the live database's most recent logged health entry -- this shows how stale (or current) the most recent backup is relative to the live data, a real operational signal rather than a fabricated one.

In [8]:
backup_files = sorted(f for f in os.listdir(BACKUPS_DIR) if f.endswith(".db"))
print(f"Backup directory      : {BACKUPS_DIR}")
print(f"Backup snapshots found: {len(backup_files)}\n")

snapshot_timestamps = []
for f in backup_files:
    full_path = os.path.join(BACKUPS_DIR, f)
    size_kb = os.path.getsize(full_path) / 1024
    mtime = datetime.fromtimestamp(os.path.getmtime(full_path))
    print(f"  {f}   {size_kb:.1f} KB   modified {mtime:%Y-%m-%d %H:%M:%S}")

    # The app's backup filenames embed a YYYYMMDD_HHMMSS timestamp -- extract it if present.
    m = re.search(r"(\d{8})_(\d{6})", f)
    if m:
        snapshot_timestamps.append(datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S"))

if snapshot_timestamps:
    print(f"\nBackup snapshots span : {min(snapshot_timestamps):%Y-%m-%d %H:%M:%S} to {max(snapshot_timestamps):%Y-%m-%d %H:%M:%S}")
print(f"Most recent live entry: {max_d}  (latest date present in registros_salud)")

conn.close()

Backup directory      : H:\DEV\HealthTrack Pro\backups
Backup snapshots found: 2

  healthtrack_backup_20260522_120438.db   96.0 KB   modified 2026-05-22 10:51:56
  healthtrack_backup_20260616_234258.db   96.0 KB   modified 2026-05-22 12:04:38

Backup snapshots span : 2026-05-22 12:04:38 to 2026-06-16 23:42:58
Most recent live entry: 2026-05-21  (latest date present in registros_salud)


## Conclusion

- The live database holds **119 real health entries** in `registros_salud`, logged across **57 distinct days** spanning **2026-03-23 to 2026-05-21** (a 60-day window, so roughly every other day has at least one entry, with several days logging multiple times a day -- `tarde` (afternoon) is the most common slot at 44 entries, followed by `noche` (night, 40) and `manana` (morning, 35)).
- Core vitals are consistently logged: systolic/diastolic BP, heart rate, oxygenation, weight, BMI and steps all have data on all 119 entries. Other fields are sparser by nature of being optional: sleep hours/quality only on 40 entries, glucose on 56, body temperature on just 32, caffeine on 55.
- Blood pressure runs on the high side across the whole window: systolic averaged **~127 mmHg** (range 94-161) and diastolic **~80 mmHg** (range 61-103). This lines up with the app's own alerting: **87 of 161 alerts (54%)** were triggered by `presion_arterial`, making it by far the most-flagged metric, ahead of stress level (29), glucose (24) and heart rate (17). Every alert in the table is marked viewed but **none are marked resolved (0 of 161)**.
- Comparing the first half of the logging window to the second half (split at the chronological midpoint, 2026-04-25) shows the direction of change for each vital -- see the "Early-vs-late trend" table above for the exact early/late averages and deltas per metric, computed directly from the real rows rather than assumed.
- Only two backup snapshots exist on disk (`healthtrack_backup_20260522_120438.db` and `healthtrack_backup_20260616_234258.db`), both 98,304 bytes. The most recent backup postdates the most recent logged health entry (2026-05-21) by roughly three and a half weeks, meaning the live app hasn't been backed up again since well before this notebook was written, and no new health entries have been logged since the last live-data date found in the table.
- Medication and symptom text fields (`medicamentos`, `sintomas`, `ejercicio_realizado`) were never used across all 119 entries; only `notas_medicas` (general notes) was used, and only in 30 of them -- so this dataset's real signal is concentrated in the numeric vitals, not the free-text fields.